# 02 · Explorar la red cinética

Inspeccionaremos los estados, las transiciones y las propensiones (tasas) del modelo. Una propensión se expresa en s⁻¹ y depende del estado actual y del ambiente.

> **ProteinMotorSim 0.1.1** · Ejemplo educativo. Las tasas predeterminadas del modelo son ilustrativas y no representan todavía una parametrización experimental validada.

## Preparación del entorno

Esta celda localiza automáticamente la raíz del repositorio e incorpora `src/` a la ruta de importación. Instala antes las dependencias con `python -m pip install -e ".[dev,viz]"` y selecciona el kernel `Python (ProteinMotorSim)`.

In [1]:
from pathlib import Path
import sys

# Busca la raíz del proyecto desde el directorio actual o sus carpetas padre.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in _candidates if (path / "pyproject.toml").exists() and (path / "src").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "No se encontró ProteinMotorSim. Abre JupyterLab desde la carpeta raíz "
        "del proyecto o desde su carpeta notebooks/."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Proyecto detectado: {PROJECT_ROOT}")

Proyecto detectado: /home/zix/ProteinMotorSim


In [2]:
from protein_motor_sim import Environment, Kinesin1

motor = Kinesin1()
network = motor.build_network()
environment = Environment(atp_concentration=1.0e-3, temperature_K=310.0)

## 1. Estados

In [3]:
for state_name, state in network.states.items():
    print(f"{state_name:20s} terminal={state.is_terminal} | {state.description}")

bound_apo            terminal=False | Motor unido al microtúbulo, sin ATP unido.
atp_bound            terminal=False | Estado efectivo con ATP unido.
post_hydrolysis      terminal=False | Estado efectivo posterior a hidrólisis.
detached             terminal=True | Motor desprendido del microtúbulo.


## 2. Transiciones

In [4]:
for transition in network.transitions:
    rate = transition.rate(environment)
    print(
        f"{transition.source:18s} -> {transition.target:18s} "
        f"| evento={transition.event_name:18s} "
        f"| tasa={rate:10.4g} s^-1 "
        f"| desplazamiento={transition.displacement_nm:6.2f} nm "
        f"| ATP={transition.atp_consumed}"
    )

bound_apo          -> atp_bound          | evento=atp_binding        | tasa=      1000 s^-1 | desplazamiento=  0.00 nm | ATP=0
atp_bound          -> post_hydrolysis    | evento=atp_hydrolysis     | tasa=        80 s^-1 | desplazamiento=  0.00 nm | ATP=1
post_hydrolysis    -> bound_apo          | evento=forward_step       | tasa=       100 s^-1 | desplazamiento=  8.20 nm | ATP=0
post_hydrolysis    -> bound_apo          | evento=backward_step      | tasa=      0.05 s^-1 | desplazamiento= -8.20 nm | ATP=0
bound_apo          -> detached           | evento=detachment         | tasa=       0.2 s^-1 | desplazamiento=  0.00 nm | ATP=0
atp_bound          -> detached           | evento=detachment         | tasa=       0.2 s^-1 | desplazamiento=  0.00 nm | ATP=0
post_hydrolysis    -> detached           | evento=detachment         | tasa=       0.2 s^-1 | desplazamiento=  0.00 nm | ATP=0


## 3. Propensiones disponibles desde cada estado

In [5]:
for state_name in network.states:
    print(f"\nEstado: {state_name}")
    for transition, rate in network.rates(state_name, environment):
        print(f"  {transition.event_name:20s}: {rate:.5g} s^-1")


Estado: bound_apo
  atp_binding         : 1000 s^-1
  detachment          : 0.2 s^-1

Estado: atp_bound
  atp_hydrolysis      : 80 s^-1
  detachment          : 0.2 s^-1

Estado: post_hydrolysis
  forward_step        : 100 s^-1
  backward_step       : 0.05 s^-1
  detachment          : 0.2 s^-1

Estado: detached


## 4. Cómo cambia la unión de ATP con la concentración

In [6]:
from dataclasses import replace

for atp_M in (1.0e-5, 1.0e-4, 1.0e-3, 5.0e-3):
    condition = replace(environment, atp_concentration=atp_M)
    atp_rate = next(
        rate for transition, rate in network.rates("bound_apo", condition)
        if transition.event_name == "atp_binding"
    )
    print(f"[ATP]={atp_M:.2e} M -> k_on[ATP]={atp_rate:.3g} s^-1")

[ATP]=1.00e-05 M -> k_on[ATP]=10 s^-1
[ATP]=1.00e-04 M -> k_on[ATP]=100 s^-1
[ATP]=1.00e-03 M -> k_on[ATP]=1e+03 s^-1
[ATP]=5.00e-03 M -> k_on[ATP]=5e+03 s^-1


## 5. Nota de modelado

La red es una representación efectiva de tres estados activos y un estado terminal. No representa por separado las dos cabezas, los sitios de unión del microtúbulo ni todos los intermediarios conformacionales de kinesina-1. Las tasas actuales son ilustrativas.